## In this exercise, you'll put together a RAG system and compare outputs from RAG vs. just querying an LLM.

## For this exercise, you'll be asking about Subspace-Constrained LoRA (SC-LoRA), a new technique described in a [recent article publised on arXiv.org](https://arxiv.org/abs/2505.23724). You've been provided the text of this article in the file 2505.23724v1.txt.

In [1]:
# !pip install faiss-cpu
# !pip install langchain
# !pip install langchain-community
# !pip install langchain-openai
# !pip install langchain-huggingface

In [2]:
import os
import json

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from openai import OpenAI

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

/tmp/ipykernel_67015/2946999983.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## First, you'll need to set up a way to interact with the generator model. You can use the OpenAI class from the openai library for this. See [this page](https://developers.openai.com/api/docs/guides/text?lang=python) for more information. When you do this, you'll need to set the base_url to "https://openrouter.ai/api/v1" and to pass in your api key. Set the model to "openrouter/owl-alpha".

In [3]:
with open("keys.txt", "r") as f:
    api_key = f.read()

In [4]:
base_url = "https://openrouter.ai/api/v1"

In [5]:
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

## First, ask the model "How does SC-LoRA differ from regular LoRA?" without providing any additional context. Read through a few different responses. Question: Are the responses accurate, or does it seem like the model is just making up something that sounds plausible?

In [6]:
model = "openrouter/owl-alpha"
query = "How does SC-LoRA differ from regular LoRA?"

In [7]:
response = client.responses.create(
    model=model,
    input=query
)

In [8]:
response.output_text

'SC-LoRA introduces a sparsity constraint to standard Low-Rank Adaptation (LoRA) to achieve more efficient and memory-friendly fine-tuning. This method dynamically prunes redundant or less important rank-one components in the low-rank updates, enhancing parameter efficiency while preserving model performance.'

# Part 1: Manual RAG
In order to get more reliable responses, let's set up a RAG system.

In this first part, you'll build all of the pieces of the RAG system individually.

First, you'll need the retriever portion. Create a FAISS index to hold the text of the article. Encode this text using the all-MiniLM-L6-v2 encoder. Note that you'll want to divide the text into smaller chunks rather than encoding the whole article all at once. You could try, for example, the [RecursiveCharacterTextSplitter class from LangChain](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter). You'll need to specify a chunk_size and chunk_overlap. You could try a chunk_size of 500 and overlap of 50 as a starting point.

In [9]:
with open("2505.23724v1.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

text_list = text_splitter.split_text(text)

In [11]:
embeddings_model = "all-MiniLM-L6-v2"

In [12]:
transformer = SentenceTransformer(embeddings_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
query_embedding = transformer.encode(query)
text_embeddings = transformer.encode(text_list)

In [14]:
index = faiss.IndexFlatIP(text_embeddings.shape[1]) # build the index
print(index.is_trained)
index.add(text_embeddings) # add vectors to the index
print(index.ntotal)

True
57


Next, use the following as a system prompt:

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)

```

Use the FAISS index to pull in relevant context to fill in the context. Try passing in this additional system prompt. Hint: you can do this by using the following messages in the client.chat.completions.create function

```
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
```

How does adding this context change the results?

In [15]:
k = 5
D, I = index.search(np.array([query_embedding]), k)

In [16]:
context = ""
for i in I[0]:
    context = context + text_list[i]

In [17]:
# Alternate
# context = '/n'.join(np.array(text_list)[I[0]])

In [18]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
  )

In [19]:
messages=[
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": query
    }
]

In [20]:
response = client.responses.create(
    model=model,
    input=messages
)

In [21]:
response.output_text

'Based on the provided context, SC-LoRA differs from regular LoRA primarily in its **initialization scheme and its focus on safety preservation**.\n\nWhile regular LoRA initializes adapter "A" with Gaussian noise and adapter "B" with a zero matrix, SC-LoRA uses a specific initialization strategy to ensure the model remains coherent with its pre-trained state. Furthermore, SC-LoRA is designed to achieve high utility while preventing **safety degradation** (catastrophic forgetting), a challenge where regular LoRA and other baselines often show notable safety drops.'

# Part 2: LangChain
You can also use the [LangChain library](https://www.langchain.com/) to help build your RAG system.

For the retriever, you can use the [HugginFaceEmbeddings class](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), using the all-MiniLM-L6-v2 model, to create your embedding model. There is also a [FAISS class](https://reference.langchain.com/python/langchain-community/vectorstores/faiss/FAISS), which has a useful from_texts method. Once you've created your vector store, use the [as_retriever method](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.as_retriever) on it and save it to a variable named retriever.

In [22]:
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
hf_embedder = HuggingFaceEmbeddings(
    model_name=embeddings_model,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
vector_store = FAISS.from_texts(text_list, hf_embedder)

In [24]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 50}
)

For the generator, you can use the [ChatOpenAI class](https://python.langchain.com/docs/integrations/chat/openai/). Be sure to set base_url="https://openrouter.ai/api/v1", model_name="openrouter/owl-alpha", and openai_api_key= Your API key. Save this to a variable named llm.

In [25]:
llm = ChatOpenAI(
    openai_api_key=api_key,
    model_name=model,
    base_url=base_url
)

Now that the two components have been created, we can combine them into a chat template using the [ChatPromptTemplate class](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html). We can set up a system prompt and the pass that in, like

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
```

In [26]:
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

Now we need to connect all of the pieces together. Newer versions of LangChain commonly use LCEL (LangChain Expression Language)[https://www.langchain.com/blog/langchain-expression-language] to build pipelines where components are connected together using the | operator.

You'll need:

A helper function that combines retrieved documents into a single string of context
A pipeline that:
extracts the question from the input
sends the question to the retriever
formats the retrieved documents into context
passes both the context and question into the prompt
sends the completed prompt to the LLM
A simplified diagram looks like this:

input ↓ extract question ↓ retrieve documents ↓ format context ↓ prompt ↓ LLM

You can create this chain by using

```
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)
```
Take a minute to study this and see if you can figure out how the syntax works.

In [27]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)

Finally, invoke your chain using:

```
response = rag_chain.invoke(
    {"question": query}
)
```

print(response.content)
Compare the output from this section with both previous approaches:

LLM without retrieval
Manual RAG
LangChain RAG
The quality of the answers from (2) and (3) should be similar. The purpose of LangChain is not to improve the model itself. Rather, it provides abstractions that simplify the retrieval and generation workflow.

In [28]:
response = rag_chain.invoke(
    {"question": query}
)

In [29]:
response.content

'Based on the provided context, SC-LoRA differs from regular LoRA primarily in its focus on **safety preservation** and its specific initialization or structural approach (indicated by the parameter $\\beta$).\n\nWhile regular LoRA (Low-Rank Adaptation) focuses on adding trainable low-rank adapters to a frozen pre-trained model, SC-LoRA is designed to achieve high utility (performance) while showing **almost no safety degradation**. The context notes that standard LoRA and other baselines often present notable safety degradation, whereas SC-LoRA maintains a lower harmfulness score and rate, especially at higher $\\beta$ values (e.g., $\\beta=0.9$).'